# RAPIDS & Scanpy Single-Cell RNA-seq Workflow on mouse NAc cells

Copyright (c) 2020, NVIDIA CORPORATION.

Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0 

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

This notebook demonstrates a single-cell RNA analysis workflow that begins with preprocessing a count matrix of size `(n_gene, n_cell)` and results in a visualization of the clustered cells for further analysis.

For demonstration purposes, we use a dataset of 1.3 M brain cells with Unified Virtual Memory to oversubscribe GPU memory.

## Import requirements

In [ ]:
import numpy as np
import scanpy as sc
import anndata
import scipy.io
import scipy.sparse

import time
import os, wget


import cudf

from cuml.decomposition import PCA
from cuml.manifold import TSNE
from cuml.cluster import KMeans
from cuml.preprocessing import StandardScaler

import cuml
import rapids_scanpy_funcs
import utils

import warnings
warnings.filterwarnings('ignore', 'Expected ')
warnings.simplefilter('ignore')
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import rmm

from rmm.allocators.cupy import rmm_cupy_allocator
import cupy
cupy.cuda.set_allocator(rmm_cupy_allocator)
from scipy import sparse
import gc
import cupy as cp
gc.collect()

import calculation_tool as ct

We use the RAPIDS memory manager to enable Unified Virtual Memory management, which allows us to oversubscribe the GPU memory.

## Input data

In the cell below, we provide the path to the sparse `.h5ad` file containing the count matrix to analyze. Please see the README for instructions on how to download the dataset we use here.

To run this notebook using your own dataset, please see the README for instructions to convert your own count matrix into this format. Then, replace the path in the cell below with the path to your generated `.h5ad` file.

## load data

In [ ]:
import os
import scanpy as sc
import pandas as pd
from scipy import sparse
import anndata

def load_and_merge_data_v3(base_dir):
    # 処理する条件（controlとschizophrenia）をリストとして定義
    conditions = ['control', 'schizophrenia']
    
    count = 0
    for condition in conditions:
        condition_path = os.path.join(base_dir, condition)
        
        # 各条件下でのサンプル名（MB7、MB8など）を取得
        samples = os.listdir(condition_path)
        
        for sample in samples:
            sample_path = os.path.join(condition_path, sample, 'matrix.tsv')
            
            # pandasでTSVファイルを読み込み
            if count == 0:
                adata=sc.read_csv(sample_path,delimiter='\t').T
                sparse_X = sparse.csr_matrix(adata.X)
                adata.X = sparse_X
                adata.obs['condition'] = condition
                adata.obs['sample'] = sample
            else:
                adata_append = sc.read_csv(sample_path,delimiter='\t').T
                sparse_X = sparse.csr_matrix(adata_append.X)
                adata_append.X = sparse_X
                adata_append.obs['condition'] = condition
                adata_append.obs['sample'] = sample
            
                # 連結
                adata = anndata.concat([adata, adata_append])
                print(adata.X.shape)
            
            count += 1
            
    return adata

base_dir = '/temp/data/human_Sz_PFC_each'
#adata = load_and_merge_data_v3(base_dir)
file_path="/temp/data/human_Sz_PFC_each/merged_adata.h5ad"
#adata.write(file_path)

In [ ]:
base_dir = '/data/human_Sz_PFC_each'
#adata = load_and_merge_data_v3(base_dir)
file_path="/data/human_Sz_PFC_each/merged_SZ_adata.h5ad"
adata = anndata.read_h5ad(file_path)
print(adata.X.shape)

In [ ]:
file_path="/data/human_Sz_PFC_each/merged_control_adata.h5ad"
adata = anndata.read_h5ad(file_path)

In [ ]:
inc_list=['MB7','MB9','MB11', 'MB13', 'MB15', 'MB16', 'MB17']

In [ ]:
adata2=adata[adata.obs['sample'].isin(inc_list)]

In [ ]:
adata2.write("/data/human_Sz_PFC_each/merged_control_selected_adata.h5ad")

In [ ]:
file_path="/data/human_Sz_PFC_each/merged_control_selected_adata.h5ad"
adata,GPCR_df=ct.preprocess_adata_in_bulk(file_path,label=None,is_gpu=False)
GPCR_df.to_csv("/data/human_Sz_PFC_each/control_selected_combined_data_GPCR_df.csv")

In [ ]:
file_path="/data/human_Sz_PFC_each/merged_control_adata.h5ad"
adata,GPCR_df=ct.preprocess_adata_in_bulk(file_path,label=None,is_gpu=False)
GPCR_df.to_csv("/data/human_Sz_PFC_each/control_combined_data_GPCR_df.csv")

In [ ]:
GPCR_df=pd.read_csv("/data/human_Sz_PFC_each/control_selected_combined_data_GPCR_df.csv")
adata=anndata.read_h5ad("/data/human_Sz_PFC_each/merged_control_selected_adata_processed.h5ad")

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
D_R_mtx,GPCR_type_df,drug_list,GPCR_list=ct.load_parameters()
params=ct.set_parameters_for_preprocess(GPCR_list)

In [ ]:
D_R_mtx,GPCR_type_df,drug_list,GPCR_list=ct.load_parameters()
params=ct.set_parameters_for_preprocess(GPCR_list)
import calculation_tool as ct
ct.drug_titeration(adata, GPCR_df, GPCR_type_df, drug_list, D_R_mtx)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=50
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
file_path="/data/human_Sz_PFC_each/merged_control_selected_adata.h5ad"
file_root, file_extension = os.path.splitext(file_path)
# Append '_processed' to the root and add the extension back
processed_file_path = f"{file_root}_processed{file_extension}"
adata.write(processed_file_path)

In [ ]:
GPCR_adata=anndata.AnnData(X=GPCR_df)
GPCR_adata_norm=sc.pp.normalize_total(GPCR_adata,target_sum=1e4,inplace=False)['X']
GPCR_adata_norm_df=pd.DataFrame(GPCR_adata_norm,columns=GPCR_adata.var.index)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_50_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=4)
dir="/data/human_Sz_PFC_each/threshold_50_r_4"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=5)
dir="/data/human_Sz_PFC_each/threshold_50_r_5"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
sc.pl.umap(adata, color=["is_clz_selective"])

In [ ]:
#import calculation_tool as ct

drug_conc=10**4
results_df_sorted,all_responses=ct.sim_inhibit_pattern(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,n_pattern=10000)

In [ ]:
dir="/data/human_Sz_PFC_each/control_selected"
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses.csv"))

In [ ]:
ct.visualize_patterns(results_df_sorted, top_n=20, top_n_for_heatmap=20, scatter_n=500)

In [ ]:
import calculation_tool as ct

drug_conc=10**4
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc)

In [ ]:
dir="/data/human_Sz_PFC_each/"
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))

In [ ]:
ct.visualize_patterns(results_df_sorted, top_n=20, top_n_for_heatmap=20, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=100
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_100_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=4)
dir="/data/human_Sz_PFC_each/threshold_100_r_4"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=500
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_500_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=4)
dir="/data/human_Sz_PFC_each/threshold_500_r_4"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=300
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_300_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=400
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_400_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=200
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_200_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=450
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_450_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=30, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
drug_conc=10**4
adata=ct.calc_drug_response(adata,GPCR_df,GPCR_type_df,drug_list,D_R_mtx,drug_conc)
selectivity_threshold=480
adata,num_clz_selective=ct.calc_clz_selective_cell(adata,drug_list,selectivity_threshold)

In [ ]:
results_df_sorted,all_responses=ct.sim_inhibit_pattern_3r(adata,GPCR_adata_norm_df,GPCR_type_df,drug_conc,group_col="is_clz_selective", selected_label=True,n_inhibited=3)
dir="/data/human_Sz_PFC_each/threshold_480_r_3"
if not os.path.exists(dir):
    os.makedirs(dir)
results_df_sorted.to_csv(os.path.join(dir,"results_df_sorted_3r.csv"))
all_responses.to_csv(os.path.join(dir,"all_responses_3r.csv"))
ct.visualize_patterns(results_df_sorted, top_n=50, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
ct.visualize_patterns(results_df_sorted, top_n=50, top_n_for_heatmap=30, scatter_n=500)

In [ ]:
ct.visualize_patterns(results_df_sorted, top_n=50, top_n_for_heatmap=50, scatter_n=200)

## 5-HT1A / M2 / H3 阻害時の予測 cAMP 比較（クロザピン反応 vs 非反応）

クロザピン反応細胞（`is_clz_selective=True`）を抽出した上で、`HTR1A`（5-HT1A）, `CHRM2`（M2）, `HRH3`（H3）を同時阻害した場合の予測 cAMP 応答を、反応群/非反応群で比較する。
検定（Welchのt検定 + Mann-Whitney U）と可視化（box/violin + swarm）を実行する。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, mannwhitneyu

# ===== 3受容体同時阻害パターン（5-HT1A, M2, H3） =====
target_pattern = {
    "HTR1A_raw": True,   # 5-HT1A inhibition
    "CHRM2_raw": True,   # M2 inhibition
    "HRH3_raw": True,    # H3 inhibition
}



def compute_camp_response_for_pattern(
    adata,
    GPCR_adata_norm_df,
    GPCR_type_df,
    drug_conc,
    pattern,
    group_col="is_clz_selective",
    selected_label=True,
    Ki_inhibited=0.01,
    Ki_not_inhibited=10000,
):
    """特定阻害パターン時の予測cAMP反応を計算（mouse notebookの処理に準拠）。"""

    Ki_df = GPCR_type_df.copy()
    Ki_df["Ki"] = Ki_not_inhibited
    for receptor_col, inhibit in pattern.items():
        if receptor_col in Ki_df.index and inhibit:
            Ki_df.loc[receptor_col, "Ki"] = Ki_inhibited

    camp = ct.calc_camp(adata, GPCR_adata_norm_df, Ki_df, drug_conc)

    df_plot = pd.DataFrame({
        "cAMP_response": camp,
        "group": np.where(adata.obs[group_col] == selected_label, "clz_selective", "non_clz_selective"),
        "leiden": adata.obs["leiden"].astype(str).values,
    }, index=adata.obs_names)

    summary = (
        df_plot.groupby("group")["cAMP_response"]
        .agg(["count", "mean", "median", "std"])
        .rename(columns={"count": "n"})
    )
    return df_plot, summary


def run_camp_analysis_with_debug(
    adata,
    GPCR_adata_norm_df,
    GPCR_type_df,
    drug_conc,
    target_pattern,
    group_col="is_clz_selective",
    selected_label=True,
):
    """進捗表示つきでcAMP比較を実行し、クラッシュ箇所を特定しやすくする。"""
    import traceback

    debug_log = []

    def _step(name, fn):
        t0 = time.time()
        print(f"[START] {name}")
        try:
            out = fn()
            dt = time.time() - t0
            print(f"[DONE ] {name} ({dt:.2f}s)")
            debug_log.append({"step": name, "status": "ok", "sec": dt})
            return out
        except Exception as e:
            dt = time.time() - t0
            print(f"[FAIL ] {name} ({dt:.2f}s): {e}")
            traceback.print_exc()
            debug_log.append({"step": name, "status": "fail", "sec": dt, "error": str(e)})
            return None

    print(f"cells={adata.n_obs}, genes={adata.n_vars}, drug_conc={drug_conc}")

    missing = [k for k in target_pattern.keys() if k not in GPCR_type_df.index]
    if missing:
        print(f"[WARN ] pattern keys missing in GPCR_type_df.index: {missing}")

    result = _step(
        "compute_camp_response_for_pattern",
        lambda: compute_camp_response_for_pattern(
            adata=adata,
            GPCR_adata_norm_df=GPCR_adata_norm_df,
            GPCR_type_df=GPCR_type_df,
            drug_conc=drug_conc,
            target_pattern=target_pattern,
            group_col=group_col,
            selected_label=selected_label,
        ),
    )
    if result is None:
        return None, None, pd.DataFrame(debug_log)

    df_plot_3r, summary_3r = result

    df_cmp = _step("dropna & split groups", lambda: df_plot_3r.dropna(subset=["cAMP_response", "group"]).copy())
    if df_cmp is None:
        return df_plot_3r, summary_3r, pd.DataFrame(debug_log)

    clz_vals = df_cmp.loc[df_cmp["group"] == "clz_selective", "cAMP_response"].values
    non_vals = df_cmp.loc[df_cmp["group"] == "non_clz_selective", "cAMP_response"].values
    print(f"n_clz_selective: {len(clz_vals)}")
    print(f"n_nonselective: {len(non_vals)}")

    _step("quick summary", lambda: print({
        "selective_mean": float(np.mean(clz_vals)) if len(clz_vals) else np.nan,
        "nonselective_mean": float(np.mean(non_vals)) if len(non_vals) else np.nan,
        "diff": (float(np.mean(clz_vals)) - float(np.mean(non_vals))) if len(clz_vals) and len(non_vals) else np.nan,
    }))

    return df_plot_3r, summary_3r, pd.DataFrame(debug_log)


# 予測cAMP応答を計算（mouse notebookの関数を利用）
df_plot_3r, summary_3r, debug_progress_df = run_camp_analysis_with_debug(
    adata=adata,
    GPCR_adata_norm_df=GPCR_adata_norm_df,
    GPCR_type_df=GPCR_type_df,
    drug_conc=drug_conc,
    target_pattern=target_pattern,
    group_col="is_clz_selective",
    selected_label=True,
)

display(debug_progress_df)
print(summary_3r)

# ===== 群比較（全細胞） =====
df_cmp = df_plot_3r.dropna(subset=["cAMP_response", "group"]).copy()
clz_vals = df_cmp.loc[df_cmp["group"] == "clz_selective", "cAMP_response"].values
non_vals = df_cmp.loc[df_cmp["group"] == "non_clz_selective", "cAMP_response"].values

welch_t = ttest_ind(clz_vals, non_vals, equal_var=False, nan_policy="omit")
mann_u = mannwhitneyu(clz_vals, non_vals, alternative="two-sided")

stats_df = pd.DataFrame({
    "test": ["Welch_ttest", "Mann_Whitney_U"],
    "statistic": [welch_t.statistic, mann_u.statistic],
    "p_value": [welch_t.pvalue, mann_u.pvalue],
    "n_clz": [len(clz_vals), len(clz_vals)],
    "n_non": [len(non_vals), len(non_vals)],
    "mean_clz": [np.mean(clz_vals), np.mean(clz_vals)],
    "mean_non": [np.mean(non_vals), np.mean(non_vals)],
    "median_clz": [np.median(clz_vals), np.median(clz_vals)],
    "median_non": [np.median(non_vals), np.median(non_vals)],
})

display(stats_df)

# ===== 可視化 =====
plot_df = df_cmp.copy()
plot_df["group"] = pd.Categorical(plot_df["group"], ["non_clz_selective", "clz_selective"])

plt.figure(figsize=(6, 5))
sns.violinplot(data=plot_df, x="group", y="cAMP_response", inner=None, cut=0, color="lightgray")
sns.boxplot(data=plot_df, x="group", y="cAMP_response", width=0.35, showcaps=True,
            boxprops={"facecolor": "white", "zorder": 3}, showfliers=False)
sns.swarmplot(data=plot_df, x="group", y="cAMP_response", size=2.3, alpha=0.5, color="black")

plt.title("Predicted cAMP under 5-HT1A/M2/H3 inhibition")
plt.xlabel("")
plt.ylabel("Predicted cAMP response")
plt.tight_layout()
plt.show()

# クラスタ別に見たい場合（任意）
plt.figure(figsize=(12, 4))
sns.boxplot(data=plot_df, x="leiden", y="cAMP_response", hue="group", showfliers=False)
plt.title("Predicted cAMP by Leiden cluster (5-HT1A/M2/H3 inhibition)")
plt.xlabel("Leiden")
plt.ylabel("Predicted cAMP response")
plt.legend(title="group", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

